# Big Chalk: Qwen2.5-Math-7B Cold-Start SFT on Kaggle Dual Tesla T4 (32 GB VRAM)

Trains `Qwen/Qwen2.5-Math-7B` base on the 605 certified reasoning traces (`data/chalk_seeds_500.jsonl`) with:
1. Multi-GPU model sharding (`device_map="auto"` putting 14 layers on GPU 0 and 14 layers + LM head on GPU 1).
2. LoRA (r=32, alpha=64) on all 7 linear projections (`q, k, v, o, gate, up, down`).
3. Strict prompt loss masking on the 5-tag reasoning schema (`<explore>`, `<conjecture>`, `<test_edge_cases>`, `<lemma_isolate>`, `<formal_proof>`).
4. Expanded context window (2048 tokens) to prevent contest proof truncation.
5. Peak VRAM ~11.5 GB per T4 (well within the 16 GB limit).

**Hardware**: Kaggle Notebook with **Accelerator: GPU T4 x2** selected in the notebook settings.

In [ ]:
# 1. Verify Dual Tesla T4 GPUs (32 GB Combined VRAM)
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU not detected! Make sure Accelerator is set to 'GPU T4 x2'."
device_count = torch.cuda.device_count()
print(f"Detected {device_count} CUDA Device(s):")
for i in range(device_count):
    vram_gb = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({vram_gb:.2f} GB VRAM)")
if device_count < 2:
    print("Warning: Only 1 GPU detected. Training 7B will require --load_in_4bit to fit in 16 GB.")

In [ ]:
# 2. Install Required Python Dependencies
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets

In [ ]:
# 3. Clone Repository and Checkout Big Chalk Branch
import os
if not os.path.exists("t4-cuda"):
    !git clone -b feat-big-chalk-math-7b https://github.com/Epoch-AI-Lab/t4-cuda.git
    %cd t4-cuda
else:
    %cd t4-cuda
    !git fetch origin
    !git checkout feat-big-chalk-math-7b
    !git pull origin feat-big-chalk-math-7b

In [ ]:
# 4. Launch Multi-GPU Cold-Start SFT Training on Qwen2.5-Math-7B
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!python3 benchmarks/train_math_sft.py \
    --model_name_or_path "Qwen/Qwen2.5-Math-7B" \
    --data_path "data/chalk_seeds_500.jsonl" \
    --output_dir "/kaggle/working/big_chalk_7b_sft" \
    --epochs 3 \
    --batch_size 1 \
    --grad_accum 16 \
    --lr 1.5e-4 \
    --max_length 2048 \
    --lora_r 32 \
    --lora_alpha 64 \
    --logging_steps 5

In [ ]:
# 5. Run External Benchmark Evaluation on Held-Out AIME and AMC 12 Contests
!python3 benchmarks/eval_math_benchmark.py \
    --model_name_or_path "Qwen/Qwen2.5-Math-7B" \
    --adapter_path "/kaggle/working/big_chalk_7b_sft/lora_adapter" \
    --benchmark_path "data/external_math_eval.json" \
    --output_path "/kaggle/working/big_chalk_7b_eval_results.json" \
    --max_new_tokens 2048

In [ ]:
# 6. Package Trained LoRA Adapter and Results for Download
!tar -czvf /kaggle/working/big_chalk_7b_adapter.tar.gz -C /kaggle/working/big_chalk_7b_sft lora_adapter
print("Artifact created at /kaggle/working/big_chalk_7b_adapter.tar.gz")